# Mooring Field Detection — Kaggle GPU Pipeline

**Before running:**
1. Settings → Accelerator → **GPU T4** (not P100)
2. Settings → Internet → **On**
3. Add-ons → Secrets → add `GOOGLE_MAPS_API_KEY`

Run cells top to bottom. Do **not** restart the kernel mid-session.

In [ ]:
# Cell 1 — Clone repo and install dependencies
# TODO: Replace YOUR_USER with your actual GitHub username before running
GITHUB_URL = "https://github.com/YOUR_USER/MooringFieldDetection.git"

import subprocess, sys
if "YOUR_USER" in GITHUB_URL:
    raise ValueError("Edit GITHUB_URL above — replace YOUR_USER with your GitHub username.")

import subprocess
subprocess.run(["git", "clone", GITHUB_URL, "/kaggle/working/MooringFieldDetection"], check=True)
%cd /kaggle/working/MooringFieldDetection

# Install packages missing from Kaggle's environment
for pkg in ["ultralytics", "httpx", "python-dotenv"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True)

# Register mooring_fields package without touching Kaggle's pre-installed libs
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], capture_output=True)
print("install done")

In [ ]:
# Cell 2 — Fix Python path + bootstrap Kaggle environment
# bootstrap_kaggle() sets CWD to repo root, detects GPU, and loads Kaggle Secrets
import sys, json
sys.path.insert(0, "/kaggle/working/MooringFieldDetection/src")

from mooring_fields.runtime import bootstrap_kaggle
info = bootstrap_kaggle()
print(json.dumps(info, indent=2))
# Expect: cuda: true, gpu: Tesla T4, secrets.GOOGLE_MAPS_API_KEY: true

In [ ]:
# Cell 3 — Load API key from Kaggle Secrets and fetch satellite imagery
# (Explicit load here is a safe fallback in case bootstrap_kaggle() didn't find it)
import os, json
from kaggle_secrets import UserSecretsClient
os.environ["GOOGLE_MAPS_API_KEY"] = UserSecretsClient().get_secret("GOOGLE_MAPS_API_KEY")

from mooring_fields.fetch_imagery import fetch_all
result = fetch_all()
print(json.dumps(result, indent=2))
# Expect: downloaded ~615 tiles (or skipped_cached ~615 if already fetched this session)

In [ ]:
# Cell 4 — Prelabel: run YOLO inference on all tiles to generate boat annotations
import json
from mooring_fields.prelabel_boats import prelabel_all
result = prelabel_all()
print(json.dumps(result, indent=2))
# Expect: complete: true for train and val splits

In [ ]:
# Cell 5 — Train: fine-tune YOLOv8l-OBB on prelabeled boat annotations
# Uses config/training.yaml settings (yolov8l-obb.pt, 150 epochs, batch_gpu=8)
# Takes ~45-90 min on T4 — epoch logs will scroll below
import json
from mooring_fields.train_boats import train
from mooring_fields.runtime import publish_outputs

report = train()
report["published"] = publish_outputs()
print(json.dumps(
    {k: v for k, v in report.items() if k != "results"},
    indent=2
))
# Key metrics: mAP50 (target >0.83), best_weights path

In [ ]:
# Cell 6 — Evaluate: detect and cluster boats on val sites, compute Hit@150m
import json
from mooring_fields.evaluate import evaluate_val
from mooring_fields.runtime import publish_outputs

report = evaluate_val()
report["published"] = publish_outputs()

# Print summary only (skip verbose per_site detail)
summary = {k: v for k, v in report.items() if k not in ("per_site", "clusters")}
print(json.dumps(summary, indent=2))
# Key metric: hit_rate_pct — % of val mooring fields correctly detected
# Outputs published to: /kaggle/working/mooring_outputs/
#   - mooring_boats/weights/best.pt   (trained model)
#   - mooring_fields.db               (boats + fields + GPS + location names)
#   - evaluation_results.json / evaluation_clusters.kml

In [ ]:
# Cell 6b — Inspect the saved database (boats + fields + GPS + location names)
import sqlite3, json
from mooring_fields.paths import DB_PATH

conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row

n_scans = conn.execute("SELECT COUNT(*) FROM scans").fetchone()[0]
n_fields = conn.execute("SELECT COUNT(*) FROM fields").fetchone()[0]
n_boats = conn.execute("SELECT COUNT(*) FROM boats").fetchone()[0]
print(f"scans={n_scans}  fields={n_fields}  boats={n_boats}\n")

print("Top fields by boat count:")
rows = conn.execute(
    "SELECT boat_count, mean_confidence, location_name, latitude, longitude "
    "FROM fields ORDER BY boat_count DESC LIMIT 15"
).fetchall()
for r in rows:
    name = r["location_name"] or "(unnamed)"
    print(f"  {r['boat_count']:>3} boats | conf {r['mean_confidence']:.2f} | "
          f"{r['latitude']:.5f}, {r['longitude']:.5f} | {name}")
conn.close()
# DB file is at data/mooring_fields.db and is copied to /kaggle/working/mooring_outputs/ by publish_outputs()

## Downloading your outputs

After **Save Version → Save & Run All** completes, your trained model and database are in the **Output** tab under `mooring_outputs/`. Three ways to get them:

1. **Browser (easiest):** Output tab → Download (`best.pt`, `mooring_fields.db`, KML).
2. **API with full scope:** run `kaggle auth login` (OAuth) locally, then `kaggle kernels output <user>/<notebook> -p <dest>`. Note: `KGAT_`-style API tokens lack the `kernels.get` scope and return 403 — use OAuth login instead.
3. **Publish as dataset:** click "New Dataset" on the output, then `kaggle datasets download <user>/<name> --unzip`.

## Optional: scan new locations worldwide

After training, you can run detection on **any coastal location worldwide**.

**Steps:**
1. Drop pins in Google Earth at candidate locations → File → Save As → KML
2. In Kaggle: Add Data → upload your KML as a new dataset
3. Run the cell below, updating the `--kml` path to match your dataset

In [ ]:
# Cell 7 (optional) — Scan new KML locations using the trained model
# Update the --kml path to point to your uploaded KML dataset
import os
from kaggle_secrets import UserSecretsClient
os.environ["GOOGLE_MAPS_API_KEY"] = UserSecretsClient().get_secret("GOOGLE_MAPS_API_KEY")

# Uncomment and edit the path below, then run:
# !python -m mooring_fields.cli scan \
#     --kml /kaggle/input/your-scan-dataset/new_locations.kml \
#     --output-dir /kaggle/working/mooring_outputs/global_scan \
#     --weights /kaggle/working/MooringFieldDetection/runs/mooring_boats/weights/best.pt \
#     --max-requests 2000
#
# Output KML: /kaggle/working/mooring_outputs/global_scan/discovered_fields.kml
# Download it from the Output tab and open in Google Earth.
print("Edit and uncomment the scan command above, then re-run this cell.")